In [1]:
!pip install -q transformers torch

In [4]:
## Cell 1: Zero-Shot vs. Few-Shot Prompting
## See how adding examples improves strict format adherence.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_text(prompt, max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 1. Zero-Shot Prompting
zero_shot_prompt = """Extract product name and sentiment from this text:
'I bought the SoundBlast Headphones yesterday and the audio quality is terrible.'
"""

print("--- ZERO-SHOT OUTPUT ---")
print(generate_text(zero_shot_prompt, max_new_tokens=50))

# 2. Few-Shot Prompting
few_shot_prompt = """Extract product name and sentiment in the format 'Product | Sentiment':

Input: The Apex Laptop is lightning fast and battery lasts all day!
Output: Apex Laptop | Positive

Input: Returned the PowerBank 5000 because it stopped charging after two days.
Output: PowerBank 5000 | Negative

Input: I bought the SoundBlast Headphones yesterday and the audio quality is terrible.
Output:"""

print("\n--- FEW-SHOT OUTPUT ---")
print(generate_text(few_shot_prompt, max_new_tokens=50))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


--- ZERO-SHOT OUTPUT ---
negative

--- FEW-SHOT OUTPUT ---
Negative


In [7]:
## Chain-of-Thought (CoT) Prompting for Logic & Math
## Language models often struggle with multi-step logic unless instructed to reason step-by-step.
# Standard Prompt (Likely to fail or guess)
standard_prompt = """Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A:"""

# Chain-of-Thought Prompt
cot_prompt = """Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A: Let's think step by step.
1. Roger starts with 5 tennis balls.
2. 2 cans with 3 tennis balls each is 2 * 3 = 6 tennis balls.
3. 5 + 6 = 11.
The answer is 11.

Q: The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?
A: Let's think step by step."""

print("--- STANDARD PROMPT OUTPUT ---")
print(generate_text(standard_prompt, max_new_tokens=50))

print("\n--- CHAIN-OF-THOUGHT OUTPUT ---")
print(generate_text(cot_prompt, max_new_tokens=100))

--- STANDARD PROMPT OUTPUT ---
He has 5 + 2 = 6 tennis balls. He has 6 x 3 = 18 tennis balls. He has 18 - 5 = 9 tennis balls. He has 9 + 2 = 10 tennis balls. He has 9 + 10 = 13 tennis

--- CHAIN-OF-THOUGHT OUTPUT ---
The cafeteria had 23 - 20 = 9 apples. They bought 9 + 6 = 19 apples. The answer is 19.


In [8]:
## System Prompting & Persona/Format Control
## Controlling tone, perspective, and output constraints using structured instructions.
from transformers import pipeline

# Load a instruction-tuned generative model (e.g., GPT-2 or small Llama/Flan model)
gen_pipeline = pipeline("text-generation", model="gpt2")

template = """System: You are an expert JSON API assistant. Respond ONLY in valid JSON.

User: Extract location and temperature from: "It is currently 72 degrees and sunny in San Francisco."

JSON Output:
{
  "location": "San Francisco",
    "temperature": "72"
}

User: Extract location and temperature from: "It is snowing heavily in Tokyo at 2 degrees."

JSON Output:"""

output = gen_pipeline(template, max_new_tokens=40, do_sample=False)
print("--- STRUCTURED JSON OUTPUT ---")
print(output[0]['generated_text'][len(template):])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- STRUCTURED JSON OUTPUT ---


{  "location": "Tokyo",

   "temperature": "72"

}

User: Extract location and temperature from: "It is raining heavily
